# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an end-to-end workflow for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library in Python. All dataset entities, such as record sets and fields, are referenced by their unique `@id` as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We will load the Croissant dataset from the provided schema URL, then print the dataset's name and description to understand its context.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n\nDescription: {metadata.description}")


## 2. Data Overview

Let's list the available record sets in this dataset, along with their `@id` and a sample of each field (using the correct `@id` references).

In [ ]:
# List all available record sets in the dataset with their @id
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- Name: {record_set['name']} | @id: {record_set['@id']}")
    field_ids = [f["@id"] for f in record_set.get('field', [])]
    print(f"  Fields (@id): {field_ids if field_ids else '[none listed]'}")

Now, we print the first record from one of the record sets for inspection (substitute the `@id` from the list above as needed):

In [ ]:
# Pick the first record set's @id for demonstration
record_sets_list = dataset.record_sets
if record_sets_list:
    first_rs_id = record_sets_list[0]['@id']
    print(f"\nSample records from record set {first_rs_id}:")
    for i, x in enumerate(dataset.records(record_set=first_rs_id)):
        print(x)
        if i >= 2:
            break
else:
    print('No record sets found in the dataset.')

## 3. Data Extraction

We will now extract all records for each record set (referenced strictly by their `@id`) and create a Pandas DataFrame for each. This allows fast inspection and further analysis.

In [ ]:
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(records)} records. Columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print(f"  No records found for {rs_id}.")
# For demonstration, display head of the first record set loaded
if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows from record set {example_rs_id}:")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field by its `@id` (from the columns above) and perform filtering and normalization. Then, we'll group by another field (also by `@id`) if suitable.

In [ ]:
# For illustration, select the first dataframe and look for numeric columns
df = None
record_set_id = None
for key, frame in dataframes.items():
    if not frame.empty:
        df = frame
        record_set_id = key
        break

if df is not None:
    numeric_fields = df.select_dtypes(include="number").columns.tolist()
    print(f"Numeric fields found: {numeric_fields}")
    # For demo, use the first numeric field, or skip section if not present
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # This is the field @id
        print(f"Proceeding with numeric field: {numeric_field_id}")
        # Filter records based on a threshold
        threshold = df[numeric_field_id].quantile(0.25)  # e.g., 25th percentile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (25th percentile): {len(filtered_df)} rows")
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"First 5 rows (with normalized column):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to group by a categorical field
        # Try to pick the first non-numeric
        candidate_groups = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        if candidate_groups:
            group_field_id = candidate_groups[0]
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping detected.")
    else:
        print('No numeric fields found for EDA.')
else:
    print('No record set with data available for EDA.')

## 5. Visualization

Visualize the distribution of the selected numeric field (by its `@id`) and, if applicable, compare it across groups defined by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If we grouped earlier, show a barplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion

In this notebook, we have:

- Loaded a complex clinical and molecular dataset using the Croissant schema and the `mlcroissant` library.
- Explored available record sets and fields by their fully qualified `@id`.
- Extracted records and demonstrated record set loading into DataFrames with field `@id` referencing.
- Applied basic EDA: filtering, normalization, and (if possible) grouping.
- Visualized the main data distributions.

This workflow can be customized for deeper domain-specific analysis or modeling according to your research needs. Always reference fields by their `@id` for clarity and interoperability. For large-scale or automated workflows, refer to the [`mlcroissant` documentation](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant) for advanced data manipulation and exporting options.